In [ ]:
import os
import pandas as pd
import numpy as np
from pandasql import sqldf

# =====================================================================
# 1. NẠP DỮ LIỆU TỪ EXCEL & DỌN DẸP TIÊU ĐỀ
# =====================================================================
current_dir = os.getcwd()
raw_file = os.path.join(current_dir, 'data', 'raw_data.xlsx')
output_file = os.path.join(current_dir, 'data', 'data_clean.csv')

df_raw = pd.read_excel(raw_file)
df_raw.columns = df_raw.columns.str.strip()

# =====================================================================
# 2. XỬ LÝ LỖI GỘP Ô NGAY TRÊN PYTHON (BẮT BUỘC TRƯỚC KHI CHẠY SQL)
# =====================================================================
# Giải thích: Nếu chạy SQL ngay, các dòng gộp ô bị trống sẽ bị SQL lọc mất hoặc không thể điền đầy.
# Vì thế, ta dùng Python điền đầy (ffill) cột ngày tháng và mã hóa đơn trước.
df_raw['Invoice'] = df_raw['Invoice'].replace(r'^\s*$', np.nan, regex=True).ffill()
df_raw['InvoiceDate'] = df_raw['InvoiceDate'].replace(r'^\s*$', np.nan, regex=True).ffill()

# Khởi tạo hàm chạy SQL nội bộ kết nối trực tiếp với bảng df_raw đã sửa lỗi gộp ô
run_sql = lambda q: sqldf(q, env={'df_raw': df_raw})


# =====================================================================
# 3. CHẠY TRUY VẤN SQL: LÀM SẠCH, ĐỔI TÊN & LỌC DỮ LIỆU LỖI
# =====================================================================
# Thực hiện bằng SQL:
# - 1.1 Loại bỏ hai cột Customer ID và Country (bằng cách không SELECT chúng)
# - 1.2 Đổi tên các trường dữ liệu theo đúng yêu cầu phân công
# - Xử lý chuyển "2,55" -> "2.55" và ép kiểu Price thành số thực, Quantity thành số nguyên
query = """ 
    SELECT
        Invoice AS invoice_no,                      -- Định danh mã đơn hàng (Giỏ hàng nhặt đồ)
        StockCode AS stock_code,                    -- Mã SKU sản phẩm (Đối tượng định vị kho)
        InvoiceDate AS order_date,                  -- Ngày giờ giao dịch
        Description AS description,                 -- Mô tả sản phẩm
        CAST(Quantity AS INTEGER) AS quantity,      -- Số lượng sản phẩm
        CAST(REPLACE(CAST(Price AS TEXT), ',', '.') AS REAL) AS price -- Đơn giá sản phẩm
    FROM df_raw
    WHERE 
        -- Lọc bỏ các dòng lỗi (Số lượng <= 0 hoặc Giá <= 0 hoặc rỗng)
        CAST(Quantity AS INTEGER) > 0 
        AND CAST(REPLACE(CAST(Price AS TEXT), ',', '.') AS REAL) > 0
        AND Price IS NOT NULL;
""" 

# Chạy SQL và lưu kết quả vào bảng df_clean
df_clean = run_sql(query)


# =====================================================================
# 4. CHUẨN HÓA CHI TIẾT NGÀY THÁNG BẰNG PYTHON
# =====================================================================
# Tách phần giờ và ép kiểu ngày tháng về định dạng chuẩn YYYY-MM-DD
df_clean['order_date'] = pd.to_datetime(df_clean['order_date'], errors='coerce').dt.date
df_clean['order_date'] = df_clean['order_date'].ffill()

# Sắp xếp lại thứ tự các cột cho chuẩn chỉnh và khoa học
cols = ['order_date', 'invoice_no', 'stock_code', 'description', 'quantity', 'price']
df_clean = df_clean[cols]


# =====================================================================
# 5. XUẤT THẲNG RA FILE CSV SẠCH (DATA_CLEAN.CSV)
# =====================================================================
df_clean.to_csv(output_file, index=False, sep=';', encoding='utf-8-sig')